# WVS Wave 7: Age and Gender Exploration

**Research focus:** Explore how age and gender relate to self-reported life satisfaction (Q49), financial satisfaction (Q50), and perceived freedom/control (Q48).

**Dataset:** WVS Wave 7 cross-sectional respondent file (97,220 rows; 613 variables).  
**Weighting:** Descriptive means use `W_WEIGHT`. The report includes unweighted sample counts (`n`) alongside weighted means. These are weighted descriptive summaries, not causal estimates or design-based confidence intervals.

**Coding choices**
- Q49, Q50, Q48: retain responses 1–10; negative/special missing codes are excluded.
- Q260: 1 = Male, 2 = Female (standard WVS coding).
- Q262: retain ages 16–110.
- Age groups: 16–29, 30–44, 45–59, 60–74, 75+.
- Country-level gender gaps are Female minus Male; positive values indicate higher female mean Q49.

**Interpretation caution:** Pooled age/gender patterns can reflect differences in country composition. The country fixed-effects model below provides an adjusted descriptive check, but it does not establish causality. WVS weighting does not by itself account for all complex survey design features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# Update this path if the CSV is stored elsewhere.
DATA_PATH = "WVS_Cross-National_Wave_7_csv_v6_0(1).csv"

raw = pd.read_csv(DATA_PATH, low_memory=False)
df = raw[['Q49','Q50','Q48','Q260','Q262','B_COUNTRY_ALPHA','W_WEIGHT']].copy()

for col in ['Q49','Q50','Q48','Q260','Q262','W_WEIGHT']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['gender'] = df['Q260'].map({1:'Male', 2:'Female'})
df['age'] = df['Q262'].where(df['Q262'].between(16, 110))
for q in ['Q49','Q50','Q48']:
    df[q] = df[q].where(df[q].between(1, 10))

age_labels = ['16–29','30–44','45–59','60–74','75+']
df['age_group'] = pd.cut(df['age'], [15,29,44,59,74,110], labels=age_labels)

def weighted_mean(data, column):
    data = data.dropna(subset=[column, 'W_WEIGHT'])
    return np.average(data[column], weights=data['W_WEIGHT'])

print('Rows:', len(df), '| Country/region codes:', df['B_COUNTRY_ALPHA'].nunique())
print('Valid Q49:', df['Q49'].notna().sum())
print('Valid age:', df['age'].notna().sum())
print('Weight range:', df['W_WEIGHT'].min(), 'to', df['W_WEIGHT'].max())

## 1. Age group patterns

In [ ]:
age_rows = []
for age_group, g in df.dropna(subset=['age_group']).groupby('age_group', observed=True):
    row = {'Age group': str(age_group), 'n Q49': g['Q49'].notna().sum()}
    for q in ['Q49','Q50','Q48']:
        row[q + ' weighted mean'] = weighted_mean(g, q)
    age_rows.append(row)
age_summary = pd.DataFrame(age_rows)
display(age_summary.round(3))

fig, ax = plt.subplots(figsize=(9,5.5))
for q, label in [('Q49','Life satisfaction (Q49)'),
                  ('Q50','Financial satisfaction (Q50)'),
                  ('Q48','Freedom/control (Q48)')]:
    vals = [weighted_mean(df[df['age_group'].astype(str)==ag], q) for ag in age_labels]
    ax.plot(age_labels, vals, marker='o', label=label)
ax.set_ylabel('Weighted mean (1–10)')
ax.set_xlabel('Age group')
ax.set_title('Age patterns across three self-reported outcomes')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=.25)
plt.tight_layout()
plt.show()

**Initial read:** Q49 and Q50 are lowest in the 45–59 group and higher in the older groups. Q48 does not follow the same pattern: it is relatively high in the 30–44 group and lower among respondents aged 75+. The 75+ group is smaller than the younger groups, so its estimate should be interpreted with that sample size in mind.

## 2. Gender differences overall and across age groups

In [ ]:
gender_rows = []
for gender, g in df.dropna(subset=['gender']).groupby('gender'):
    gender_rows.append({
        'Gender': gender, 'n Q49': g['Q49'].notna().sum(),
        'Q49 weighted mean': weighted_mean(g,'Q49'),
        'Q50 weighted mean': weighted_mean(g,'Q50'),
        'Q48 weighted mean': weighted_mean(g,'Q48')
    })
gender_summary = pd.DataFrame(gender_rows)
display(gender_summary.round(3))

age_gender_rows = []
for ag in age_labels:
    row = {'Age group': ag}
    for gender in ['Female','Male']:
        g = df[(df['age_group'].astype(str)==ag) & (df['gender']==gender)]
        row[gender] = weighted_mean(g, 'Q49')
    row['Female−Male'] = row['Female'] - row['Male']
    age_gender_rows.append(row)
age_gender = pd.DataFrame(age_gender_rows)
display(age_gender.round(3))

fig, ax = plt.subplots(figsize=(8.5,5.2))
pos = np.arange(len(age_gender)); width = .36
ax.bar(pos-width/2, age_gender['Female'], width, label='Female')
ax.bar(pos+width/2, age_gender['Male'], width, label='Male')
ax.set_xticks(pos, age_gender['Age group'])
ax.set_ylabel('Weighted mean Q49 (1–10)')
ax.set_title('Life satisfaction by age group and gender')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=.25)
plt.tight_layout()
plt.show()

**Initial read:** The overall weighted Q49 difference is small. In the pooled age groups, women have slightly higher Q49 means in the younger and middle-age groups, while men have higher means in the 60–74 and 75+ groups. These pooled contrasts may partly reflect country composition; the next section checks country-level variation.

## 3. Country-level gender differences

In [ ]:
country_rows = []
valid = df.dropna(subset=['Q49','gender','B_COUNTRY_ALPHA','W_WEIGHT'])
for country, g in valid.groupby('B_COUNTRY_ALPHA'):
    f = g[g['gender']=='Female']; m = g[g['gender']=='Male']
    country_rows.append({
        'Country code': country,
        'Female n': len(f), 'Male n': len(m),
        'Female mean': weighted_mean(f,'Q49'),
        'Male mean': weighted_mean(m,'Q49'),
        'Female−Male': weighted_mean(f,'Q49') - weighted_mean(m,'Q49')
    })
country_gap = pd.DataFrame(country_rows).sort_values('Female−Male')
display(country_gap.head(8).round(3))
display(country_gap.tail(8).round(3))
print('Countries/regions:', len(country_gap))
print('Median country gap:', country_gap['Female−Male'].median().round(3))
print('Country gap range:', country_gap['Female−Male'].min().round(3), 'to', country_gap['Female−Male'].max().round(3))

fig, ax = plt.subplots(figsize=(10,9))
ax.barh(country_gap['Country code'], country_gap['Female−Male'])
ax.axvline(0, linewidth=1)
ax.set_xlabel('Female − Male weighted mean Q49')
ax.set_title('Country-level gender gap in life satisfaction')
ax.grid(axis='x', alpha=.25)
plt.tight_layout()
plt.show()

**Initial read:** The direction and magnitude of the gender gap vary substantially across country/region samples. The pooled difference should therefore not be interpreted as a uniform cross-national pattern. Country codes are used here; attach the WVS country-code lookup when preparing a public-facing chart.

## 4. Country fixed-effects adjusted descriptive model

In [ ]:
# Weighted model: categorical age groups × gender, controlling for country fixed effects.
# This is an adjusted descriptive comparison, not a causal model or design-based inference.
model_data = df.dropna(subset=['Q49','age_group','gender','B_COUNTRY_ALPHA','W_WEIGHT']).copy()
model = smf.wls(
    'Q49 ~ C(age_group) * C(gender) + C(B_COUNTRY_ALPHA)',
    data=model_data, weights=model_data['W_WEIGHT']
).fit()
print('Weighted R-squared:', round(model.rsquared, 4))

# Standardize predicted age/gender means to the same observed weighted country distribution.
country_weight = model_data.groupby('B_COUNTRY_ALPHA')['W_WEIGHT'].sum()
countries = country_weight.index.tolist()
standardized = []
for ag in age_labels:
    for gender in ['Female','Male']:
        new = pd.DataFrame({
            'age_group': ag, 'gender': gender,
            'B_COUNTRY_ALPHA': countries
        })
        pred = model.predict(new)
        standardized.append({
            'Age group': ag, 'Gender': gender,
            'Country-standardized predicted Q49': np.average(pred, weights=country_weight.values)
        })
standardized = pd.DataFrame(standardized)
display(standardized.round(3))

## Summary and limitations

1. **Age:** Weighted Q49 and Q50 show a midlife low and higher means in older age groups; Q48 has a different age pattern.
2. **Gender:** The pooled Q49 gap is small, but the age-group pattern and country-level gaps are heterogeneous.
3. **Country composition:** Country fixed effects and common-country-distribution predictions help check pooled differences, but do not solve all comparability issues.
4. **Weights:** Results use WVS `W_WEIGHT`. For publication-quality inference, verify the Wave 7 weighting documentation and survey design variables, and consider design-based standard errors.
5. **Scope:** These are cross-sectional associations. They do not show that age, gender, financial satisfaction, or perceived freedom causes life satisfaction.